[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/03_otel_sdk_python/03_proyecto_otel_sdk_python.ipynb)

# Proyecto 03 · Instrumenta a mano SupportOps v0 y mira tus primeras trazas

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Opcional (`--profile core`) para el último paso; el resto se verifica en memoria. Sin Docker funciona igual (modo degradado) |
| **RAM aproximada** | < 0,5 GB (con `core` en reposo ≈ 0,9 GB) |
| **Prerrequisitos** | Lección `03_otel_sdk_python` (spans, contexto, procesadores, instrumentación manual) y módulo 01 (`Instrumentor`) |
| **Conexión** | Parte de **SupportOps v0** (módulo 01, sin observabilidad) y entrega **SupportOps v1 instrumentado a mano**. El módulo 04 lo ajustará a las GenAI semantic conventions |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. Hoy, cuando un cliente se queja de que «el bot tardó mucho», nadie sabe si fue el modelo, una herramienta o un reintento: el agente v0 es una caja negra.
Tu primera tarea es instrumentarlo **a mano** con el SDK de OpenTelemetry (sin autoinstrumentación) para que cada ticket produzca **una traza** con el árbol `invoke_agent → chat / execute_tool`, con errores marcados,
tokens anotados y **logs correlacionados**. Después usarás tus propias trazas para contestar tres preguntas de operación.

## Datos

Tráfico sintético de SupportOps (semilla fija `42`, esquema en `schema.TICKET_SCHEMA`): **60 tickets** con fallos de fondo (`DEFAULT_FAULT_RATES`), tiempo **simulado**. Cadena de resiliencia: stack local (último paso) → memoria (toda la verificación) → no hace falta nada pregrabado.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución, `mock`, semilla 42, 60 tickets) |
|---|---|---|---|
| R1 | `init_tracing` | `Resource` con `service.name = supportops-agent`, `service.version` y `service.instance.id` | cumple |
| R2 | Una traza por ticket | Cada ticket produce **1 traza** con **1 raíz** `invoke_agent <agente>` | 60/60 (100 %) |
| R3 | Spans = pasos | Spans de la traza = `1 + n_llm_calls + n_tool_calls` | 243 spans en total (122 `chat` + 61 `execute_tool` + 60 raíces), 100 % de trazas |
| R4 | Jerarquía y *kinds* | Todo `chat`/`execute_tool` es hijo directo de la raíz; raíz `INTERNAL`, `chat` `CLIENT`, herramienta `INTERNAL` | 100 % |
| R5 | Tiempos | Duración de la raíz = `AgentResult.latency_s` (±1 ms) | 100 % |
| R6 | Errores | Nº de spans hijos en `ERROR` = pasos con error (5); raíz en `ERROR` ⇔ estado `error`/`max_steps` (1 raíces); los spans en error llevan `error.type` y un evento `exception` | 100 % |
| R7 | Atributos | `chat`: `gen_ai.operation.name`, `gen_ai.request.model`, `supportops.attempt` y tokens; suma de tokens de los `chat` = tokens del `AgentResult` | 100 % |
| R8 | Logs correlacionados | Cada log del agente (125 en total) lleva el `trace_id` y `span_id` **de su traza** | 100 % |
| R9 | Paralelismo | 3 herramientas en hilos → **1 traza** (no 4) y 3 hijos de la raíz | cumple |
| R10 | Operación con trazas | Tus respuestas (reintentos = 1, herramienta más lenta y su p95) coinciden con la verdad de terreno | exacto |

> 📏 Todo es **estructural y determinista** porque el tiempo es simulado y la semilla fija: no depende de la velocidad de tu máquina. Los umbrales del 100 % son lo que mide la solución de referencia;
> con tráfico distinto (otra semilla) los totales cambian pero las proporciones deben seguir siendo 100 %.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import contextvars, json, secrets, threading, time, logging, inspect, urllib.request, http.server
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from opentelemetry import trace, context, propagate
from opentelemetry.trace import SpanKind, Status, StatusCode
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider, sampling
from opentelemetry.sdk.trace.export import (BatchSpanProcessor, SimpleSpanProcessor, ConsoleSpanExporter, SpanExporter, SpanExportResult)
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from supportops.telemetry import setup_telemetry, OTelInstrumentor

STACK = check_stack(["collector", "tempo", "loki"])
logging.getLogger("opentelemetry").setLevel(logging.ERROR)
N = 60
tickets = generate_tickets(N, seed=SEED, fault_rates=DEFAULT_FAULT_RATES, start_s=SIM_EPOCH_S)    # tiempo simulado fijo → rúbrica determinista
print(len(tickets), "tickets ·", sum(bool(t.faults) for t in tickets), "con fallos inyectados")

## Paso 1 · El proveedor: `init_tracing`

**TODO 1.** Implementa `init_tracing(service_name, otlp_endpoint=None)` que devuelva `(provider, memory_exporter)`:

- un `Resource` con `service.name`, `service.version="0.1.0"` y un `service.instance.id` único;
- un `TracerProvider` con ese resource;
- una **copia en memoria** (`InMemorySpanExporter` + `SimpleSpanProcessor`) para que podamos verificar (solo para el laboratorio);
- **si** `otlp_endpoint` no es `None`: además un `BatchSpanProcessor` con `OTLPSpanExporter` (la URL completa es `<endpoint>/v1/traces`).

<details>
<summary>🪜 Pista 1</summary>

Es la celda 4.1 y la sección 5 de la lección: `Resource.create({...})`, `TracerProvider(resource=...)`, `provider.add_span_processor(...)`. Uno de los procesadores es Simple (memoria) y el otro Batch (OTLP).

</details>

<details>
<summary>🪜 Pista 2</summary>

`from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter` y `OTLPSpanExporter(endpoint=f"{otlp_endpoint.rstrip('/')}/v1/traces")`. Para el id único: `f"{service_name}-{secrets.token_hex(4)}"`.

</details>

In [ ]:
def init_tracing(service_name: str = "supportops-agent", otlp_endpoint: str | None = None):
    # TODO 1
    raise NotImplementedError("TODO 1")

try:
    provider, mem = init_tracing(); print(provider.resource.attributes["service.name"], "·", provider.resource.attributes.get("service.instance.id"))
except NotImplementedError as e:
    provider = mem = None; print("Pendiente:", e)

## Paso 2 · El instrumentor

Implementarás `StudentInstrumentor(Instrumentor)`. El agente llama a tus *context managers* alrededor de cada trabajo; **la excepción que ocurra dentro del bloque `with` atraviesa tu `yield`** y el agente la captura fuera (puedes registrarla y volver a lanzarla).
Usa el **reloj simulado** (`ctx.clock.now_ns()`) como `start_time`/`end_time` para que las trazas sean deterministas.

**TODO 2 · `_span`.** El ayudante común: abre un span con `start_span(..., start_time=...)`, lo **activa** con `trace.use_span(..., end_on_exit=False, record_exception=False, set_status_on_exception=False)`, y:

- si el cuerpo lanza una excepción: `record_exception` (con `timestamp` simulado), atributo `error.type` (usa `e.error_type` si existe, si no el nombre de la clase), `set_status(Status(StatusCode.ERROR, ...))` y **relanza**;
- al final (siempre): `span.end(end_time=...)` con el reloj simulado.

<details>
<summary>🪜 Pista 1</summary>

Es el patrón de `ManualInstrumentor._span` de la lección. Los atributos de **muestreo** (nombre de operación, modelo, herramienta) pásalos en `attributes=` al crear. `try/except BaseException: ...; raise` y `finally: span.end(...)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`end_time=max(ctx.clock.now_ns(), t0)` evita duraciones negativas. `str(getattr(e, 'error_type', type(e).__name__))` da el `error.type` (`ToolError` y `LLMError` tienen `error_type`).

</details>

**TODO 3 · `agent_run`.** Span raíz `invoke_agent <ctx.agent_name>`, `SpanKind.INTERNAL`, atributos al crear: `gen_ai.operation.name="invoke_agent"`, `gen_ai.agent.name`, `supportops.ticket.id`, `supportops.customer.tier`.
Al terminar (`ctx.result` ya está disponible tras el `yield`): `supportops.status`, `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens`; y si `ctx.result.status` es `error` o `max_steps`: `error.type` y estado `ERROR` (un fallo **recuperado** con reintento no marca la raíz).

**TODO 4 · `llm_call`.** Span `chat <info.model>`, `SpanKind.CLIENT`, atributos al crear: `gen_ai.operation.name="chat"`, `gen_ai.provider.name`, `gen_ai.request.model`, `supportops.attempt` (= `info.attempt`) y `supportops.step`. Tras el `yield`, si hay `info.response`: `gen_ai.usage.input_tokens/output_tokens` y `gen_ai.response.model`.
Si el LLM falla, `info.error` es un `LLMError` con `.status` (HTTP): añade `http.response.status_code`.

**TODO 5 · `tool_call`.** Span `execute_tool <info.call.name>`, `SpanKind.INTERNAL`, atributos al crear: `gen_ai.operation.name="execute_tool"`, `gen_ai.tool.name`, `gen_ai.tool.call.id`, `supportops.step`.

**TODO 6 · `log`.** `log(ctx, level, message, fields)` debe **añadir a `self.logs`** un diccionario `{"body", "severity", "trace_id", "span_id", "attributes"}` con el `trace_id` (32 hex) y `span_id` (16 hex) **del span activo en ese instante** (`trace.get_current_span().get_span_context()`).

<details>
<summary>🪜 Pista 1</summary>

`ctx_ = trace.get_current_span().get_span_context()` y `format(ctx_.trace_id, '032x')`, `format(ctx_.span_id, '016x')`. Ojo: si no hay span activo, `trace_id == 0`.

</details>

In [ ]:
class StudentInstrumentor(Instrumentor):
    def __init__(self, tracer, log_sink: list):
        self.tracer, self.logs = tracer, log_sink

    @contextmanager
    def _span(self, ctx, name, kind, attrs):
        # TODO 2: abre el span, actívalo, gestiona errores y ciérralo con el reloj simulado; haz `yield span`
        raise NotImplementedError("TODO 2")
        yield

    @contextmanager
    def agent_run(self, ctx):
        # TODO 3
        raise NotImplementedError("TODO 3")
        yield

    @contextmanager
    def llm_call(self, ctx, info):
        # TODO 4
        raise NotImplementedError("TODO 4")
        yield

    @contextmanager
    def tool_call(self, ctx, info):
        # TODO 5
        raise NotImplementedError("TODO 5")
        yield

    def log(self, ctx, level, message, fields):
        # TODO 6
        raise NotImplementedError("TODO 6")


def instrument_and_run(tickets_, factory=None):
    """Ejecuta el agente con tu instrumentor sobre un proveedor nuevo. Devuelve (resultados, spans_terminados, logs)."""
    prov, mem_ = init_tracing()
    logs: list = []
    instr = StudentInstrumentor(prov.get_tracer("supportops.student", "0.1.0"), logs)
    res = run_tickets(SupportAgent(instrumentor=instr), tickets_)
    return res, mem_.get_finished_spans(), logs

try:
    results, spans, logs = instrument_and_run(tickets)
    print(f"{len(results)} tickets → {len(spans)} spans · {len({s.context.trace_id for s in spans})} trazas · {len(logs)} logs")
except NotImplementedError as e:
    results, spans, logs = [], [], []; print("Pendiente (completa los TODO 1-6):", e)

### Mira una traza

Dibuja la traza del ticket más lento. Si todo va bien verás `invoke_agent` con sus `chat`/`execute_tool` hijos; si ves **barras sueltas sin padre** o un solo span, revisa el TODO 2 (¿activaste el span con `use_span`?).

In [ ]:
def to_records(spans_):
    return [{"trace_id": format(s.context.trace_id, "032x"), "span_id": format(s.context.span_id, "016x"), "parent_span_id": format(s.parent.span_id, "016x") if s.parent else None,
             "name": s.name, "kind": s.kind.name, "start_unix_nano": s.start_time, "end_unix_nano": s.end_time, "status": s.status.status_code.name} for s in spans_]

if spans:
    recs = to_records(spans); df = pd.DataFrame(recs); df["dur"] = (df.end_unix_nano - df.start_unix_nano) / 1e9
    tid = df[df.parent_span_id.isna()].sort_values("dur").iloc[-1].trace_id
    fig, ax = plt.subplots(figsize=(11, 0.45 * (df.trace_id == tid).sum() + 1.4)); fig.subplots_adjust(left=0.3)
    viz.waterfall([r for r in recs if r["trace_id"] == tid], ax=ax, title=f"Tu traza más lenta · {tid[:8]}…"); plt.show()
else:
    print("Aún no hay spans: completa los TODO.")

> 👀 **Qué debes observar**
>
> - Debe verse **una raíz** `invoke_agent …` con barras hijas `chat …` y `execute_tool …` **debajo y dentro** de su intervalo, no solapadas entre sí (el agente es secuencial).
> - Una barra **roja** hija con la raíz en verde es un fallo *recuperado* (p. ej. reintento tras un 5xx): la raíz no se marca. Si la raíz sí es roja, el agente se rindió (`error`/`max_steps`).
> - Barras sueltas sin padre o una traza de un solo span indican que el span no se activó con `use_span` (TODO 2) o que la raíz y los hijos usan tracers de proveedores distintos.

## Paso 3 · Herramientas en paralelo sin romper la traza

Algunos agentes ejecutan varias herramientas a la vez. **TODO 7.** Implementa `run_tools_parallel(tracer, tool_names)`: crea un span raíz `invoke_agent parallel-demo` y, **dentro**, lanza **una tarea por herramienta** en un `ThreadPoolExecutor`; cada tarea abre un span `execute_tool <nombre>`.
El resultado debe ser **una sola traza** con la raíz y tantos hijos como herramientas (en la lección viste qué pasa si no copias el contexto).

<details>
<summary>🪜 Pista 1</summary>

Los hilos nuevos empiezan con el contexto vacío. Hay que ejecutar la tarea **dentro de una copia del contexto actual**: `contextvars.copy_context().run(fn, *args)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`futs = [ex.submit(contextvars.copy_context().run, work, name) for name in tool_names]` y luego `[f.result() for f in futs]`. La copia se hace **en el hilo que llama** (el que tiene el span raíz activo), no dentro de la tarea.

</details>

In [ ]:
def run_tools_parallel(tracer, tool_names) -> None:
    # TODO 7
    raise NotImplementedError("TODO 7")

try:
    _p, _m = init_tracing("parallel-demo"); run_tools_parallel(_p.get_tracer("demo"), ["get_order", "search_kb", "check_stock"])
    _s = _m.get_finished_spans(); print("trazas:", len({s.context.trace_id for s in _s}), "· spans:", len(_s))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · Operar con tus propias trazas

Con la telemetría que acabas de producir, **responde sin mirar los `AgentResult`** (solo los `spans` terminados):

**TODO 8a.** `retries_from_spans(spans) -> int`: ¿cuántos **reintentos** de llamada al LLM hubo? (spans `chat` con `supportops.attempt >= 1`).
**TODO 8b.** `slowest_tool(spans) -> (nombre, p95_s)`: de las herramientas (`gen_ai.tool.name`), la de mayor **percentil 95** de duración (usa `viz.percentile` y la duración del span en segundos).

<details>
<summary>🪜 Pista 1</summary>

`s.attributes.get(...)`, `(s.end_time - s.start_time) / 1e9`. Agrupa las duraciones por `s.attributes['gen_ai.tool.name']` con un diccionario de listas.

</details>

In [ ]:
def retries_from_spans(spans_) -> int:
    # TODO 8a
    raise NotImplementedError("TODO 8a")

def slowest_tool(spans_):
    # TODO 8b
    raise NotImplementedError("TODO 8b")

try:
    print("reintentos:", retries_from_spans(spans), "· herramienta más lenta (p95):", slowest_tool(spans))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 5 · Rúbrica

Ejecuta la celda para puntuar tu trabajo. Las comprobaciones comparan tus spans con la **verdad de terreno** (`AgentResult`).

In [ ]:
def run_rubric():
    rows = []
    def add(r, desc, ok, detail=""): rows.append({"criterio": r, "descripción": desc, "resultado": "✅ OK" if ok else "❌ FALLA", "detalle": detail})
    try:
        res, sp, lg = instrument_and_run(tickets)
        prov, _ = init_tracing(); a = prov.resource.attributes
        add("R1", "Resource con service.name/version/instance.id", a.get("service.name") == "supportops-agent" and "service.version" in a and "service.instance.id" in a)
    except Exception as e:
        add("R1", "init_tracing + instrumentor", False, repr(e)[:90]); return pd.DataFrame(rows).set_index("criterio")
    by = {}
    for s in sp: by.setdefault(format(s.context.trace_id, "032x"), []).append(s)
    roots = {t: [s for s in v if s.parent is None] for t, v in by.items()}
    ok2 = len(by) == len(res) and all(len(r) == 1 and r[0].name.startswith("invoke_agent ") for r in roots.values())
    add("R2", "1 traza y 1 raíz por ticket", ok2, f"{len(by)} trazas / {len(res)} tickets")
    # Asociamos cada resultado con su traza por el atributo supportops.ticket.id de la raíz
    tr_of = {r[0].attributes.get("supportops.ticket.id"): t for t, r in roots.items() if r}
    def spans_of(r): return by.get(tr_of.get(r.ticket_id), [])
    add("R3", "spans = 1 + n_llm + n_tool en el 100 % de trazas", all(len(spans_of(r)) == 1 + r.n_llm_calls + r.n_tool_calls for r in res), f"{len(sp)} spans")
    def hier_ok(r):
        S = spans_of(r); root = [s for s in S if s.parent is None]
        if len(root) != 1: return False
        rid = root[0].context.span_id
        return (root[0].kind == SpanKind.INTERNAL and all(s.parent.span_id == rid for s in S if s is not root[0] and s.parent is not None)
                and all((s.kind == SpanKind.CLIENT) == s.name.startswith("chat ") for s in S if s is not root[0]) and all(s.parent is not None for s in S if s is not root[0]))
    add("R4", "jerarquía y kinds", all(hier_ok(r) for r in res))
    def dur_ok(r):
        root = [s for s in spans_of(r) if s.parent is None]
        return bool(root) and abs((root[0].end_time - root[0].start_time) / 1e9 - r.latency_s) < 1e-3
    add("R5", "duración de la raíz = latency_s (±1 ms)", all(dur_ok(r) for r in res))
    child_err = sum(s.status.status_code == StatusCode.ERROR for s in sp if s.parent is not None)
    truth_err = sum(st.status == "error" for r in res for st in r.steps)
    root_err_ok = all((([s for s in spans_of(r) if s.parent is None][0].status.status_code == StatusCode.ERROR) == (r.status in ("error", "max_steps"))) for r in res if spans_of(r))
    detail_ok = all("error.type" in s.attributes and any(e.name == "exception" for e in s.events) for s in sp if s.parent is not None and s.status.status_code == StatusCode.ERROR)
    add("R6", "errores: spans ERROR = pasos con error, raíz coherente, error.type + exception", child_err == truth_err and root_err_ok and detail_ok, f"{child_err} vs {truth_err}")
    def attrs_ok(r):
        chats = [s for s in spans_of(r) if s.name.startswith("chat ")]
        need = ("gen_ai.operation.name", "gen_ai.request.model", "supportops.attempt")
        return (all(all(k in s.attributes for k in need) for s in chats)
                and sum(s.attributes.get("gen_ai.usage.input_tokens", 0) for s in chats) == r.input_tokens
                and sum(s.attributes.get("gen_ai.usage.output_tokens", 0) for s in chats) == r.output_tokens)
    add("R7", "atributos de chat y tokens = AgentResult", all(attrs_ok(r) for r in res))
    valid_tr = set(by)
    good_logs = [l for l in lg if l.get("trace_id") in valid_tr and len(l.get("span_id") or "") == 16]
    add("R8", "logs con trace_id/span_id de su traza", len(lg) > 0 and len(good_logs) == len(lg), f"{len(good_logs)}/{len(lg)}")
    try:
        p2, m2 = init_tracing("par"); run_tools_parallel(p2.get_tracer("t"), ["a", "b", "c"]); s2 = m2.get_finished_spans()
        roots2 = [s for s in s2 if s.parent is None]
        add("R9", "3 herramientas en paralelo = 1 traza, 3 hijos de la raíz", len({s.context.trace_id for s in s2}) == 1 and len(roots2) == 1 and sum(s.parent is not None and s.parent.span_id == roots2[0].context.span_id for s in s2) == 3)
    except Exception as e:
        add("R9", "paralelismo", False, repr(e)[:90])
    try:
        steps = [st for r in res for st in r.steps]
        truth_retries = sum(1 for st in steps if st.kind == "llm" and st.attributes.get("attempt", 0) >= 1)
        by_tool: dict = {}
        for st in steps:
            if st.kind == "tool": by_tool.setdefault(st.name, []).append(st.end_s - st.start_s)
        truth_tool = max(((n, viz.percentile(v, .95)) for n, v in by_tool.items()), key=lambda x: x[1])
        got_r, got_t = retries_from_spans(sp), slowest_tool(sp)
        add("R10", "reintentos y herramienta más lenta (p95) = verdad", got_r == truth_retries and got_t[0] == truth_tool[0] and abs(got_t[1] - truth_tool[1]) < 1e-6, f"tú: {got_r}, {got_t[0]} · verdad: {truth_retries}, {truth_tool[0]}")
    except Exception as e:
        add("R10", "operación con trazas", False, repr(e)[:90])
    return pd.DataFrame(rows).set_index("criterio")

display(run_rubric())

## Paso 6 · (Opcional) Ver tus trazas en Tempo

Con el stack arriba (`docker compose --profile core up -d` desde `stack/`), exporta por OTLP y busca una traza.
**TODO 9.** Implementa `export_live(n=20)`: genera `n` tickets con `start_s="live"` (la ventana debe terminar *ahora*, o Tempo rechazará los tiempos), ejecútalos con tu instrumentor sobre `init_tracing(otlp_endpoint="http://localhost:4318")`, haz `force_flush()` y devuelve el `trace_id` (32 hex) de la primera traza. Luego la celda siguiente la busca en Tempo.

<details>
<summary>🪜 Pista 1</summary>

`prov.force_flush()` antes de consultar; el `trace_id` está en `spans[0].context.trace_id` (`format(..., '032x')`). Recuerda `prov.shutdown()` al final.

</details>

In [ ]:
def export_live(n: int = 20) -> str:
    # TODO 9
    raise NotImplementedError("TODO 9")

try:
    if STACK.degraded:
        print("Modo degradado: levanta el stack para este paso →", STACK.command)
    else:
        TID = export_live(); print("trace_id enviado:", TID)
        for _ in range(20):
            try:
                d = backends.tempo_trace(TID); n_sp = sum(len(ss["spans"]) for b in d.get("batches", []) for ss in b.get("scopeSpans", []))
                if n_sp: print("Tempo devolvió", n_sp, "spans de tu traza ✅"); break
            except backends.BackendError: pass
            time.sleep(2)
        else:
            print("Aún no está en Tempo (puede tardar ~30 s): reintenta la celda.")
except NotImplementedError as e:
    print("Pendiente:", e)

---
## Retos extra (nivel experto)

1. **Muestreo:** monta un proveedor con `ParentBased(TraceIdRatioBased(0.25))`, pasa 400 tickets y mide qué fracción de trazas conserva. ¿Cuántos *fallos* conserva? ¿Y si quieres conservar **todos** los fallos? (pista: no se puede decidir al crear el span → *tail sampling*, módulo 18).
2. **Links de reintento:** haz que el `chat` del reintento lleve un `trace.Link` al span del intento fallido (guárdalo en `ctx.state`). ¿Cómo se ve en Tempo?
3. **Límites:** provoca un `tool_loop` (`FaultSpec("tool_loop")`) y comprueba cuántos spans genera. Configura `SpanLimits(max_events=...)` y razona qué se pierde primero si añades un evento por paso.
4. **Configuración por entorno:** reescribe `init_tracing` para que no reciba argumentos y se configure solo con `OTEL_SERVICE_NAME`, `OTEL_RESOURCE_ATTRIBUTES` y `OTEL_EXPORTER_OTLP_ENDPOINT` (pista: el SDK las lee si no pasas el argumento).
5. **Fork-safe:** diseña (sin ejecutar Gunicorn) en qué hook inicializarías `init_tracing` y cómo darías un `service.instance.id` distinto a cada worker.

## Reflexión (producción e hilo conductor)

- Tu instrumentor usa **tiempos simulados**. ¿Qué dos cosas cambiarías para un servicio real (pista: `start_time`, `end_time`, y qué procesador)?
- Un fallo **recuperado** (reintento exitoso) deja la raíz sin `ERROR`. ¿Es lo que quiere el equipo de guardia? ¿Cómo te enterarías de que el proveedor LLM está fallando si todas las raíces salen verdes? (Spoiler: spans hijos `ERROR` y métricas; módulos 09 y 16.)
- ¿Qué atributos del span `execute_tool` serían **peligrosos** si contuvieran los argumentos completos de `issue_refund`? (módulo 20)
- En el módulo 04 renombrarás atributos según la spec GenAI. ¿Cuáles de los tuyos (`supportops.*` vs `gen_ai.*`) crees que **no** cambiarán y por qué?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable. Al final vuelve a ejecutar la rúbrica con ella.

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
# TODO 1
def init_tracing(service_name: str = "supportops-agent", otlp_endpoint: str | None = None):
    resource = Resource.create({"service.name": service_name, "service.version": "0.1.0", "service.instance.id": f"{service_name}-{secrets.token_hex(4)}"})
    provider = TracerProvider(resource=resource)
    mem_exporter = InMemorySpanExporter()
    provider.add_span_processor(SimpleSpanProcessor(mem_exporter))                      # copia local solo para verificar (en producción: Batch)
    if otlp_endpoint:
        from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
        provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(endpoint=f"{otlp_endpoint.rstrip('/')}/v1/traces")))
    return provider, mem_exporter


class StudentInstrumentor(Instrumentor):
    def __init__(self, tracer, log_sink: list):
        self.tracer, self.logs = tracer, log_sink

    # TODO 2
    @contextmanager
    def _span(self, ctx, name, kind, attrs):
        t0 = ctx.clock.now_ns()
        span = self.tracer.start_span(name, kind=kind, attributes=attrs, start_time=t0)           # atributos de muestreo AL CREAR
        try:
            with trace.use_span(span, end_on_exit=False, record_exception=False, set_status_on_exception=False):
                yield span
        except BaseException as e:
            span.record_exception(e, timestamp=ctx.clock.now_ns())
            span.set_attribute("error.type", str(getattr(e, "error_type", type(e).__name__)))
            span.set_status(Status(StatusCode.ERROR, str(e)[:120]))
            raise
        finally:
            span.end(end_time=max(ctx.clock.now_ns(), t0))

    # TODO 3
    @contextmanager
    def agent_run(self, ctx):
        t = ctx.ticket
        attrs = {"gen_ai.operation.name": "invoke_agent", "gen_ai.agent.name": ctx.agent_name, "supportops.ticket.id": t.ticket_id, "supportops.customer.tier": t.tier}
        with self._span(ctx, f"invoke_agent {ctx.agent_name}", SpanKind.INTERNAL, attrs) as span:
            yield
            r = ctx.result
            if r is not None:
                span.set_attributes({"supportops.status": r.status, "gen_ai.usage.input_tokens": r.input_tokens, "gen_ai.usage.output_tokens": r.output_tokens})
                if r.status in ("error", "max_steps"):
                    span.set_attribute("error.type", r.error_type or r.status)
                    span.set_status(Status(StatusCode.ERROR, r.status))

    # TODO 4
    @contextmanager
    def llm_call(self, ctx, info):
        attrs = {"gen_ai.operation.name": "chat", "gen_ai.provider.name": info.provider, "gen_ai.request.model": info.model,
                 "supportops.attempt": info.attempt, "supportops.step": info.step}
        with self._span(ctx, f"chat {info.model}", SpanKind.CLIENT, attrs) as span:
            try:
                yield
            except BaseException as e:
                if getattr(e, "status", None): span.set_attribute("http.response.status_code", int(e.status))
                raise
            if info.response is not None:
                span.set_attributes({"gen_ai.usage.input_tokens": info.response.usage.input_tokens, "gen_ai.usage.output_tokens": info.response.usage.output_tokens,
                                     "gen_ai.response.model": info.response.model})

    # TODO 5
    @contextmanager
    def tool_call(self, ctx, info):
        attrs = {"gen_ai.operation.name": "execute_tool", "gen_ai.tool.name": info.call.name, "gen_ai.tool.call.id": info.call.id, "supportops.step": info.step}
        with self._span(ctx, f"execute_tool {info.call.name}", SpanKind.INTERNAL, attrs):
            yield

    # TODO 6
    def log(self, ctx, level, message, fields):
        sc = trace.get_current_span().get_span_context()
        self.logs.append({"body": message, "severity": logging.getLevelName(level),
                          "trace_id": format(sc.trace_id, "032x") if sc.trace_id else None, "span_id": format(sc.span_id, "016x") if sc.span_id else None,
                          "attributes": dict(fields)})


# TODO 7
def run_tools_parallel(tracer, tool_names) -> None:
    def work(name):
        with tracer.start_as_current_span(f"execute_tool {name}"):
            time.sleep(0.001)
    with tracer.start_as_current_span("invoke_agent parallel-demo"):
        with ThreadPoolExecutor(max(1, len(tool_names))) as ex:
            futs = [ex.submit(contextvars.copy_context().run, work, n) for n in tool_names]       # la copia se hace AQUÍ, con el raíz activo
            [f.result() for f in futs]


# TODO 8
def retries_from_spans(spans_) -> int:
    return sum(1 for s in spans_ if s.name.startswith("chat ") and s.attributes.get("supportops.attempt", 0) >= 1)

def slowest_tool(spans_):
    d: dict = {}
    for s in spans_:
        if s.attributes.get("gen_ai.tool.name"):
            d.setdefault(s.attributes["gen_ai.tool.name"], []).append((s.end_time - s.start_time) / 1e9)
    return max(((n, viz.percentile(v, 0.95)) for n, v in d.items()), key=lambda x: x[1])


# TODO 9
def export_live(n: int = 20) -> str:
    prov, mem_ = init_tracing(otlp_endpoint="http://localhost:4318")
    instr = StudentInstrumentor(prov.get_tracer("supportops.student"), [])
    run_tickets(SupportAgent(instrumentor=instr), generate_tickets(n, seed=SEED, fault_rates=DEFAULT_FAULT_RATES, start_s="live"))
    prov.force_flush(10000)
    tid = format(mem_.get_finished_spans()[0].context.trace_id, "032x")
    prov.shutdown()
    return tid

In [ ]:
# Re-ejecuta tu comprobación con la solución de referencia
results, spans, logs = instrument_and_run(tickets)
display(run_rubric())
print("reintentos:", retries_from_spans(spans), "· herramienta más lenta (p95):", slowest_tool(spans))